# Vacancy and Substitutional defect pair in GaN

Following the publication:

> **G. Miceli, A. Pasquarello**
> "Self-compensation due to point defects in Mg-doped GaN"
> Phys. Rev. B 93, 165207 (2016)
> [DOI: 10.1103/PhysRevB.93.165207](https://doi.org/10.1103/PhysRevB.93.165207)

Focusing on recreating the defect pair in GaN from the FIG. 2(c):

<img src="https://github.com/Exabyte-io/documentation/raw/12617167278ae3523adc028583b21ea4e8ebd197/images/tutorials/materials/defects/defect_point_pair_gallium_nitride/0-figure-from-manuscript.webp" width="400" />

## 1. Prepare the Environment
### 1.1. Set up defects parameters 
Defect Configuration parameters are described in [Defect Configuration](https://github.com/Exabyte-io/made/blob/8196b759242551c77d1791bf5bd2f4150763cfef/src/py/mat3ra/made/tools/build/defect/configuration.py#L102).

In [ ]:
SUPERCELL_MATRIX = [[3, 0, 0], [0, 3, 0], [0, 0, 2]]

DEFECT_CONFIGS = [
    {"type": "substitution", "coordinate": [0.5, 0.5, 0.5], "element": "Mg", "placement_method": "closest_site"},
    {"type": "vacancy", "coordinate": [0.5, 0.5, 0.65], "placement_method": "closest_site"},
]

### 1.2. Install Packages
The step executes only in Pyodide environment. For other environments, the packages should be installed via `pip install` (see [README](../../README.ipynb)).

In [ ]:
from mat3ra.notebooks_utils.packages import install_packages

await install_packages("made|specific_examples")

### 1.3. Get input materials
Materials are loaded with `get_materials()`.

In [ ]:
from mat3ra.standata.materials import Materials
from mat3ra.made.material import Material
from mat3ra.notebooks_utils.ipython.entity.material.visualize import visualize_materials as visualize

material = Material.create(Materials.get_by_name_first_match("GaN"))
visualize(material)

### 1.4. Create and preview Supercell

In [ ]:
from mat3ra.made.tools.helpers import create_supercell

supercell = create_supercell(material, supercell_matrix=SUPERCELL_MATRIX)
visualize(supercell, repetitions=[1, 1, 1], rotation="-90x")

## 2. Create the Defect

In [ ]:
from mat3ra.made.tools.build.defective_structures.zero_dimensional.point_defect.types import PointDefectDict
from mat3ra.made.tools.helpers import create_multiple_defects

defect_dicts = [PointDefectDict(**{
    "type": defect_config["type"],
    "coordinate": defect_config["coordinate"],
    "element": defect_config.get("element", None),
    "placement_method": defect_config["placement_method"]
}) for defect_config in DEFECT_CONFIGS]

material_with_defect = create_multiple_defects(
    supercell,
    defect_dicts=defect_dicts,
)

## 3. Visualize Result(s)

In [ ]:
from mat3ra.notebooks_utils.ipython.entity.material.visualize import visualize_materials as visualize

visualize([{"material": supercell, "title": "Original material"},
           {"material": material_with_defect, "title": f"Material with defect"}],
          viewer="wave")

## 4. Pass data to the outside runtime

In [ ]:
from mat3ra.notebooks_utils.io import download_content_to_file
from mat3ra.notebooks_utils.material import set_materials

supercell.name = "GaN 3x3x2"
material_with_defect.name = "GaN 3x3x2 Mg_Ga-V_N axial pair"
set_materials([supercell, material_with_defect])
download_content_to_file(material_with_defect.to_json(), f"{material_with_defect.name}.json")